# 01 — Bell States & GHZ Preparation

**quantum-nonlocality** | Jinwon Yoo · C. Praise Anyanwu

---

This notebook covers Phase 1 of the project:

1. The four Bell states — construction, verification, and measurement
2. GHZ state preparation for arbitrary n qubits
3. Statevector inspection and entanglement verification

These are the two core entangled states our project is built on:
- The **Bell states** are the *target output* at the boundary polarization qubits
- The **GHZ state** is the *bulk resource* in the time-bin degree of freedom

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector, DensityMatrix, partial_trace, entropy
from qiskit.visualization import plot_histogram, plot_bloch_multivector

simulator = AerSimulator()
print("Imports OK ✓")

## 1. The Four Bell States

The Bell states form a maximally entangled orthonormal basis for a two-qubit system:

$$|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$$
$$|\Phi^-\rangle = \frac{1}{\sqrt{2}}(|00\rangle - |11\rangle)$$
$$|\Psi^+\rangle = \frac{1}{\sqrt{2}}(|01\rangle + |10\rangle)$$
$$|\Psi^-\rangle = \frac{1}{\sqrt{2}}(|01\rangle - |10\rangle)$$

Our project targets $|\Phi^+\rangle$ and $|\Psi^-\rangle$ as deterministic outputs at the boundary polarization qubits.

In [ ]:
def bell_state(state='phi_plus'):
    """
    Build a Bell state circuit.
    state: 'phi_plus', 'phi_minus', 'psi_plus', 'psi_minus'
    """
    qc = QuantumCircuit(2)
    qc.h(0)
    qc.cx(0, 1)
    if state == 'phi_minus':
        qc.z(0)
    elif state == 'psi_plus':
        qc.x(1)
    elif state == 'psi_minus':
        qc.x(1)
        qc.z(0)
    return qc


bell_names = ['phi_plus', 'phi_minus', 'psi_plus', 'psi_minus']
latex_names = ['|Φ+⟩', '|Φ-⟩', '|Ψ+⟩', '|Ψ-⟩']

for name, latex in zip(bell_names, latex_names):
    sv = Statevector(bell_state(name))
    print(f"{latex}: {np.round(sv.data, 4)}")

In [ ]:
# Visualize Φ+ circuit
bell_state('phi_plus').draw('mpl')

## 2. Verify Entanglement via von Neumann Entropy

For a maximally entangled two-qubit state, the von Neumann entropy of either subsystem is 1 ebit.

$$S(\rho_A) = -\text{Tr}(\rho_A \log_2 \rho_A) = 1$$

A product (unentangled) state has $S = 0$.

In [ ]:
print("Von Neumann entropy of qubit 0 (should be 1.0 for all Bell states):\n")

for name, latex in zip(bell_names, latex_names):
    sv = Statevector(bell_state(name))
    dm = DensityMatrix(sv)
    # Trace out qubit 1 to get reduced density matrix of qubit 0
    rho_A = partial_trace(dm, [1])
    S = entropy(rho_A, base=2)
    print(f"  {latex}: S = {S:.4f} ebit")

# Compare with a product state
qc_product = QuantumCircuit(2)
qc_product.h(0)  # |+⟩|0⟩ — unentangled
sv_p = Statevector(qc_product)
rho_p = partial_trace(DensityMatrix(sv_p), [1])
print(f"\n  |+⟩|0⟩ (product): S = {entropy(rho_p, base=2):.4f} ebit")

## 3. Measure Bell States — Correlations

When we measure a Bell state, the two qubits are always perfectly correlated (or anti-correlated). This is the experimental signature of entanglement.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3))

for i, (name, latex) in enumerate(zip(bell_names, latex_names)):
    qc = bell_state(name)
    qc.measure_all()
    compiled = transpile(qc, simulator)
    counts = simulator.run(compiled, shots=2048).result().get_counts()
    plot_histogram(counts, ax=axes[i], title=latex)

plt.tight_layout()
plt.suptitle('Bell state measurement outcomes (2048 shots each)', y=1.02)
plt.show()

**Key observation:**
- $|\Phi^+\rangle$, $|\Phi^-\rangle$: only $|00\rangle$ and $|11\rangle$ — correlated
- $|\Psi^+\rangle$, $|\Psi^-\rangle$: only $|01\rangle$ and $|10\rangle$ — anti-correlated

The phase difference ($+$ vs $-$) is invisible in the computational basis — it only shows up when measuring in a rotated basis (relevant for Bell inequality tests in Phase 4).

## 4. GHZ State Preparation

The GHZ state for $n$ qubits is:
$$|\text{GHZ}_n\rangle = \frac{1}{\sqrt{2}}(|0\rangle^{\otimes n} + |1\rangle^{\otimes n})$$

This is the bulk resource in our project — encoded in the **time-bin degree of freedom** of single photons.

**Circuit:** Hadamard on qubit 0, then CNOT from qubit 0 to each subsequent qubit.

In [ ]:
def ghz_state(n):
    """Prepare an n-qubit GHZ state."""
    qc = QuantumCircuit(n)
    qc.h(0)
    for i in range(1, n):
        qc.cx(0, i)
    return qc


# Visualize for n=4
ghz_state(4).draw('mpl')

In [ ]:
# Verify statevector for n=3
sv_ghz3 = Statevector(ghz_state(3))
print("GHZ(3) statevector:")
print(np.round(sv_ghz3.data, 4))
print("\nNon-zero amplitudes:")
for i, amp in enumerate(sv_ghz3.data):
    if abs(amp) > 1e-6:
        print(f"  |{format(i, f'0{3}b')}⟩: {amp:.4f}")

In [ ]:
# Measure GHZ — only |000...0⟩ and |111...1⟩ should appear
fig, axes = plt.subplots(1, 3, figsize=(12, 3))

for i, n in enumerate([2, 3, 4]):
    qc = ghz_state(n)
    qc.measure_all()
    compiled = transpile(qc, simulator)
    counts = simulator.run(compiled, shots=2048).result().get_counts()
    plot_histogram(counts, ax=axes[i], title=f'GHZ({n})')

plt.tight_layout()
plt.suptitle('GHZ state measurements — only |00…0⟩ and |11…1⟩', y=1.02)
plt.show()

## 5. GHZ Entanglement Structure

For a GHZ state, the entanglement is *global* — no pair of qubits is maximally entangled on its own. The entropy of any single qubit subsystem is 1 ebit, but tracing out all but two qubits gives a mixed (not entangled) state.

This is a key difference from a cluster state and is important for how our MBQC protocol works.

In [ ]:
n = 4
sv_ghz = Statevector(ghz_state(n))
dm_ghz = DensityMatrix(sv_ghz)

print(f"GHZ({n}) — von Neumann entropy of each single-qubit subsystem:\n")
for qubit in range(n):
    # Trace out all qubits except 'qubit'
    others = [i for i in range(n) if i != qubit]
    rho_i = partial_trace(dm_ghz, others)
    S = entropy(rho_i, base=2)
    print(f"  Qubit {qubit}: S = {S:.4f} ebit")

print("\nAll qubits have S=1 — each is maximally mixed when the others are ignored.")
print("This reflects the global, non-bipartite nature of GHZ entanglement.")

## Summary

| State | Structure | Role in project |
|---|---|---|
| Bell states | 2-qubit maximal entanglement | Target output at boundaries |
| GHZ state | n-qubit global entanglement | Bulk time-bin resource |

**Next:** `02_hybrid_state_construction.ipynb` — stitch the GHZ bulk to the boundary polarization qubits via CNOT gates and build the full hybrid entangled state.